# Network Topology Study Analysis

This notebook imports data collected by Daphne's topology study, analyzes
the effects of different network topologies on message propagation efficiency,
network load distribution, and overall system performance.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Enable copy-on-write to follow pandas recommendation.
# see https://pandas.pydata.org/docs/user_guide/copy_on_write.html
pd.options.mode.copy_on_write = True

In [ ]:
# Load collected data from Parquet file.
# Replace '../data.parquet' with the path to your data file.
# As a reminder, please run `go run ./daphne study topology` to produce this 
# file. Data is loaded in chunks and processed to categorize columns to limit 
# peak memory usage.
data = pd.read_parquet('../../data.parquet')

# Convert categorical columns for memory efficiency
for col in ['mark', 'from', 'to', 'type', 'Topology', 'Broadcast']:
    if col in data.columns:
        data[col] = data[col].astype('category')

## 2. Time to Finality

The following chart compares the time-to-finality distribution across different
network topologies. It uses the following definition of time-to-finality:

> The time from submitting the transaction to the first time the transaction got
confirmed in a block by any node.

This analysis helps understand how network topology affects transaction confirmation latency.

In [ ]:
# Filter only relevant events for TTF calculation
tx_submitted = data[data['mark'] == 'TxSubmitted'][['timestamp', 'hash', 'rid']]
tx_finalized = data[data['mark'] == 'TxFinalized'][['timestamp', 'hash', 'rid']]

# Get first TxFinalized per transaction (by hash and rid)
tx_finalized_first = tx_finalized.sort_values('timestamp').drop_duplicates(['hash', 'rid'], keep='first')

# Merge submitted and finalized to compute TTF
ttf = pd.merge(
    tx_submitted,
    tx_finalized_first,
    on=['hash', 'rid'],
    suffixes=('_submitted', '_finalized')
)
ttf['TTF'] = (ttf['timestamp_finalized'] - ttf['timestamp_submitted']) / 1e9  # convert ns to seconds

# Add Topology info by merging with StudyStarted rows
run_info = data[data['mark'] == 'StudyStarted'][['rid', 'Topology']]
ttf = pd.merge(ttf, run_info, on='rid', how='left')

# Drop missing Topology (shouldn't happen, but just in case)
ttf = ttf.dropna(subset=['Topology'])

# Calculate mean TTF per topology for sorting
ttf_means = ttf.groupby('Topology', observed=True)['TTF'].mean().sort_values()

# Plot box plot
plt.figure(figsize=(14, 7))
sns.boxplot(
    data=ttf,
    x='Topology',
    y='TTF',
    order=ttf_means.index,
    showmeans=True,
    meanprops=dict(marker='D', markerfacecolor='red', markeredgecolor='red', markersize=6)
)
plt.ylabel('Time to Finality (seconds)')
plt.xlabel('Network Topology')
plt.title('TTF Distribution by Network Topology')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 3. Total Messages by Topology

This section analyzes the total number of messages transferred across the network
for different topology types. This gives us an overview of network efficiency.

In [ ]:
# Filter message sent and study started rows
msgsent = data[data['mark'] == 'MsgSent']
scenarios = data[data['mark'] == 'StudyStarted'][['rid', 'Topology']]

# Count messages per experiment run
msg_counts = msgsent.groupby('rid').size().reset_index(name='MsgCount')

# Join with experiment parameters
merged = pd.merge(msg_counts, scenarios, on='rid')

# Group by Topology, calculate mean and std
stats = merged.groupby('Topology', observed=True)['MsgCount'].agg(['mean', 'std', 'count']).reset_index()
stats = stats.sort_values('mean')

# Plot bar chart with error bars
plt.figure(figsize=(12, 6))
plt.bar(range(len(stats)), stats['mean'], yerr=stats['std'], capsize=5, alpha=0.7)
plt.xticks(range(len(stats)), stats['Topology'], rotation=45, ha='right')
plt.title('Total Messages Transferred by Network Topology')
plt.xlabel('Network Topology')
plt.ylabel('Mean Number of Messages')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 4. Per-Node Message Distribution

This section examines how messages are distributed across individual nodes
for different topology types. This reveals load balancing characteristics.

In [ ]:
# Get message sent counts per node
msg_sent = data[data['mark'] == 'MsgSent']
msg_counts_per_node = msg_sent.groupby(['rid', 'from'], observed=True).size().reset_index(name='MsgSentCount')

# Merge with topology info
msg_counts_per_node = msg_counts_per_node.merge(scenarios, on='rid', how='left')

# Create box plot
plt.figure(figsize=(14, 7))
sns.boxplot(
    data=msg_counts_per_node,
    x='Topology',
    y='MsgSentCount',
    order=stats['Topology']  # Use same order as previous chart
)
plt.xticks(rotation=45, ha='right')
plt.title('Distribution of Messages Sent per Node by Topology')
plt.xlabel('Network Topology')
plt.ylabel('Messages Sent per Node')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Get message received counts per node
msg_received = data[data['mark'] == 'MsgReceived']
msg_recv_per_node = msg_received.groupby(['rid', 'to'], observed=True).size().reset_index(name='MsgReceivedCount')

# Merge with topology info
msg_recv_per_node = msg_recv_per_node.merge(scenarios, on='rid', how='left')

# Create box plot
plt.figure(figsize=(14, 7))
sns.boxplot(
    data=msg_recv_per_node,
    x='Topology',
    y='MsgReceivedCount',
    order=stats['Topology']  # Use same order as previous chart
)
plt.xticks(rotation=45, ha='right')
plt.title('Distribution of Messages Received per Node by Topology')
plt.xlabel('Network Topology')
plt.ylabel('Messages Received per Node')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 5. Network Bandwidth Usage

Analyze the total data volume (in bytes) transferred across different topologies.

In [ ]:
# Sum bytes per experiment run
msg_bytes = msgsent.groupby('rid')['bytesize'].sum().reset_index(name='TotalBytes')

# Join with topology info
merged_bytes = pd.merge(msg_bytes, scenarios, on='rid')

# Group by Topology
stats_bytes = merged_bytes.groupby('Topology', observed=True)['TotalBytes'].agg(['mean', 'std', 'count']).reset_index()
stats_bytes = stats_bytes.sort_values('mean')

# Convert to MB for readability
stats_bytes['mean_mb'] = stats_bytes['mean'] / (1024 * 1024)
stats_bytes['std_mb'] = stats_bytes['std'] / (1024 * 1024)

# Plot
plt.figure(figsize=(12, 6))
plt.bar(range(len(stats_bytes)), stats_bytes['mean_mb'], yerr=stats_bytes['std_mb'], capsize=5, alpha=0.7)
plt.xticks(range(len(stats_bytes)), stats_bytes['Topology'], rotation=45, ha='right')
plt.title('Total Data Transferred by Network Topology')
plt.xlabel('Network Topology')
plt.ylabel('Mean Total Data (MB)')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Message Type Breakdown

Analyze how different topologies affect the distribution of message types.

In [ ]:
# Get message counts by type and topology
msg_by_type = msgsent[msgsent['type'].notna()].groupby(['rid', 'type'], observed=True).size().reset_index(name='Count')
msg_by_type = msg_by_type.merge(scenarios, on='rid', how='left')

# Pivot to get message types as columns
pivoted = msg_by_type.pivot_table(
    index='Topology',
    columns='type',
    values='Count',
    fill_value=0,
    observed=True,
    aggfunc='mean'
)

# Sort by total messages
pivoted['total'] = pivoted.sum(axis=1)
pivoted = pivoted.sort_values('total')
pivoted = pivoted.drop(columns=['total'])

# Plot stacked bar chart
ax = pivoted.plot(kind='bar', stacked=True, figsize=(14, 7), alpha=0.8)
plt.title('Message Type Distribution by Network Topology')
plt.xlabel('Network Topology')
plt.ylabel('Mean Number of Messages')
plt.legend(title='Message Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 7. Transaction Processing Pipeline by Topology

This section analyzes the transaction processing pipeline across different topologies, showing the duration of each pipeline stage and how topology affects processing efficiency.

In [ ]:
# Filter pipeline data with all transaction marks
# Check which transaction marks are actually present
all_tx_marks = data[data['mark'].str.startswith('Tx', na=False)]['mark'].unique()

# Filter for transaction-related marks
tx_marks = ['TxSubmitted', 'TxAddedToPool', 'TxConfirmed', 'TxBeginProcessing', 'TxEndProcessing', 'TxFinalized']
available_marks = [m for m in tx_marks if m in all_tx_marks]

# Select only the columns we need for pipeline analysis
pipeline_data = data[data['mark'].isin(available_marks)][['timestamp', 'mark', 'hash', 'rid', 'Topology']].copy()

# Note: Topology is only present on StudyStarted marks, so we need to merge it
# Get topology information from StudyStarted events
study_started = data[data['mark'] == 'StudyStarted'][['rid', 'Topology']]

# Merge to add Topology to pipeline data
pipeline_data = pd.merge(
    pipeline_data.drop(columns=['Topology']),
    study_started,
    on='rid',
    how='left'
)
pipeline_data = pipeline_data.dropna(subset=['Topology'])

In [ ]:
# Calculate pipeline stage durations for each transaction
pipeline_stages = {}

# Get first occurrence of each mark per transaction
for mark_name in available_marks:
    stage_data = pipeline_data[pipeline_data['mark'] == mark_name].sort_values('timestamp').groupby(['hash', 'rid'], as_index=False).first()
    pipeline_stages[mark_name] = stage_data[['hash', 'rid', 'timestamp', 'Topology']]

# Define potential stage transitions
stage_transitions = [
    ('TxSubmitted', 'TxAddedToPool', 'Submitted→AddedToPool'),
    ('TxAddedToPool', 'TxConfirmed', 'AddedToPool→Confirmed'),
    ('TxConfirmed', 'TxBeginProcessing', 'Confirmed→BeginProcessing'),
    ('TxBeginProcessing', 'TxEndProcessing', 'BeginProcessing→EndProcessing'),
    ('TxEndProcessing', 'TxFinalized', 'EndProcessing→Finalized'),
]

# Filter to only include transitions where both marks are available
valid_transitions = [
    (start, end, name) for start, end, name in stage_transitions
    if start in available_marks and end in available_marks
]

# Find transactions that have all available stages and calculate durations
all_stage_durations = []

for rid in pipeline_data['rid'].unique():
    # Get all transactions for this run that have all available marks
    run_hashes = None
    for mark_name in available_marks:
        mark_hashes = set(pipeline_stages[mark_name][pipeline_stages[mark_name]['rid'] == rid]['hash'])
        if run_hashes is None:
            run_hashes = mark_hashes
        else:
            run_hashes = run_hashes.intersection(mark_hashes)
    
    if not run_hashes:
        continue
    
    # Calculate stage durations for complete transactions in this run
    for tx_hash in run_hashes:
        tx_stages = {}
        topology = None
        for mark_name in available_marks:
            tx_data = pipeline_stages[mark_name][(pipeline_stages[mark_name]['hash'] == tx_hash) & 
                                                   (pipeline_stages[mark_name]['rid'] == rid)]
            if not tx_data.empty:
                tx_stages[mark_name] = tx_data.iloc[0]['timestamp']
                if topology is None:
                    topology = tx_data.iloc[0]['Topology']
        
        # Calculate durations for valid transitions (timestamps are in nanoseconds)
        durations = {
            'Topology': topology,
            'rid': rid,
            'hash': tx_hash,
        }
        
        for start_mark, end_mark, transition_name in valid_transitions:
            if start_mark in tx_stages and end_mark in tx_stages:
                durations[transition_name] = (tx_stages[end_mark] - tx_stages[start_mark]) / 1e9
        
        if topology is not None and len(durations) > 3:  # Has more than just metadata
            all_stage_durations.append(durations)

if all_stage_durations:
    stage_duration_df = pd.DataFrame(all_stage_durations)
    # Get list of stage columns (exclude metadata columns)
    stage_cols = [name for _, _, name in valid_transitions if name in stage_duration_df.columns]
else:
    stage_duration_df = pd.DataFrame()
    stage_cols = []

In [ ]:
# Visualize pipeline stages with measurable durations by topology
if stage_cols:
    # Identify stages with non-zero durations
    stages_to_plot = [stage for stage in stage_cols if (stage_duration_df[stage] > 0).any()]
    
    if not stages_to_plot:
        print("No pipeline stages have measurable durations in this data.")
    else:
        # Calculate mean for topology ordering
        topology_order = stage_duration_df.groupby('Topology', observed=True)[stages_to_plot[0]].mean().sort_values().index
        
        # Determine grid layout
        n_stages = len(stages_to_plot)
        n_cols = min(2, n_stages)
        n_rows = (n_stages + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 6 * n_rows))
        if n_stages == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, stage in enumerate(stages_to_plot):
            ax = axes[idx]
            
            sns.boxplot(
                data=stage_duration_df,
                x='Topology',
                y=stage,
                order=topology_order,
                ax=ax,
                showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markeredgecolor='red', markersize=5)
            )
            ax.set_title(f'{stage}', fontsize=12, fontweight='bold')
            ax.set_xlabel('Network Topology', fontsize=10)
            ax.set_ylabel('Duration (seconds)', fontsize=10)
            ax.tick_params(axis='x', rotation=45, labelsize=9)
            ax.tick_params(axis='y', labelsize=9)
            ax.grid(axis='y', linestyle=':', alpha=0.5)
        
        # Hide extra subplots if any
        if n_stages > 1:
            for idx in range(n_stages, len(axes)):
                axes[idx].set_visible(False)
        
        title = 'Pipeline Stage Durations by Network Topology'
        if len(stages_to_plot) < len(stage_cols):
            title += f' (showing {len(stages_to_plot)} of {len(stage_cols)} stages with non-zero durations)'
        plt.suptitle(title, fontsize=14, y=0.995)
        plt.tight_layout()
        plt.show()

In [ ]:
# Summary: Mean stage durations by topology
if stage_cols:
    # Identify stages with non-zero durations
    mean_durations = stage_duration_df.groupby('Topology', observed=True)[stage_cols].mean()
    non_zero_stages = [col for col in stage_cols if mean_durations[col].max() > 0]
    
    if not non_zero_stages:
        print("All pipeline stages have zero duration - no meaningful timing differences to analyze.")
    elif len(non_zero_stages) == 1:
        # For a single stage, show a bar chart instead of stacked
        mean_durations_filtered = mean_durations[non_zero_stages].sort_values(by=non_zero_stages[0])
        
        ax = mean_durations_filtered.plot(
            kind='bar',
            figsize=(12, 6),
            color='lightcoral',
            alpha=0.8,
            legend=False
        )
        plt.title(f'Mean {non_zero_stages[0]} Duration by Network Topology', fontsize=13)
        plt.xlabel('Network Topology', fontsize=11)
        plt.ylabel('Mean Duration (seconds)', fontsize=11)
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()
    else:
        # Multiple stages - show stacked bar chart
        mean_durations_filtered = mean_durations[non_zero_stages]
        mean_durations_filtered['Total'] = mean_durations_filtered.sum(axis=1)
        mean_durations_filtered = mean_durations_filtered.sort_values('Total')
        mean_durations_filtered = mean_durations_filtered.drop(columns=['Total'])
        
        colors = ['lightcoral', 'bisque', 'lightgreen', 'lightblue', 'plum']
        stage_colors = colors[:len(non_zero_stages)]
        
        ax = mean_durations_filtered.plot(
            kind='bar',
            stacked=True,
            figsize=(14, 8),
            color=stage_colors,
            alpha=0.8
        )
        
        plt.title('Mean Pipeline Stage Durations by Network Topology', fontsize=13)
        plt.xlabel('Network Topology', fontsize=11)
        plt.ylabel('Mean Duration (seconds)', fontsize=11)
        plt.legend(title='Pipeline Stage', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()